In [19]:
# %pip install kaggle
# %pip install -U tqdm ipywidgets

In [3]:
import sys
from pathlib import Path
import subprocess
from tqdm import tqdm
from pathlib import Path
import zipfile
import zlib
import tarfile
import shutil
import time
import requests
from PIL import Image
import random
import re
from huggingface_hub import hf_hub_download
import hashlib

In [21]:
# !{sys.executable} -m kaggle auth login --force

# Download Img Net Datase


In [22]:

IMAGENET_DIR = Path("datasets/imagenet")
IMAGENET_DIR.mkdir(parents=True, exist_ok=True)

ZIP_FILE = IMAGENET_DIR / "imagenet-object-localization-challenge.zip"

TOTAL_SIZE = 168 * 1024**3   # approximate


def current_download_size():
    return sum(
        f.stat().st_size
        for f in IMAGENET_DIR.iterdir()
        if f.is_file()
    )


print("Starting ImageNet download...")

process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "kaggle",
        "competitions",
        "download",
        "imagenet-object-localization-challenge",
        "-p",
        str(IMAGENET_DIR),
    ],

    # IMPORTANT:
    # Do not use PIPE here, because an unread pipe can fill and block Kaggle.
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

previous_size = current_download_size()

with tqdm(
    total=TOTAL_SIZE,
    initial=previous_size,
    unit="B",
    unit_scale=True,
    unit_divisor=1024,
    desc="Downloading ImageNet",
    bar_format=(
        "{desc}: {percentage:6.2f}%|{bar}| "
        "{n_fmt}/{total_fmt} "
        "[{elapsed}<{remaining}, {rate_fmt}]"
    )
) as pbar:

    while process.poll() is None:

        current_size = current_download_size()
        increase = current_size - previous_size

        if increase > 0:
            pbar.update(increase)

        previous_size = current_size

        time.sleep(1)


if process.returncode != 0:
    raise RuntimeError(
        "Kaggle download failed. Run the Kaggle command directly "
        "to see the error message."
    )

print("\nDownload finished.")

Starting ImageNet download...



Download finished.


In [23]:
from pathlib import Path
import zipfile

zip_file = Path("datasets/imagenet/imagenet-object-localization-challenge.zip")

print(f"Size: {zip_file.stat().st_size / 1024**3:.2f} GiB")
print("Valid ZIP:", zipfile.is_zipfile(zip_file))

Size: 155.06 GiB
Valid ZIP: True


unzip dataset

In [24]:

# ============================================================
# CONFIG
# ============================================================

IMAGENET_DIR = Path("datasets/imagenet")
ZIP_FILE = IMAGENET_DIR / "imagenet-object-localization-challenge.zip"

# Only extract the parts we need for ImageNet-1K
KEEP_PREFIXES = (
    "ILSVRC/Data/CLS-LOC/train/",
    "ILSVRC/Data/CLS-LOC/val/",
)

KEEP_FILES = {
    "LOC_synset_mapping.txt",
    "LOC_val_solution.csv",
}


# ============================================================
# SAFE PATH CHECK
# ============================================================

def safe_target(base, filename):
    base = base.resolve()
    target = (base / filename).resolve()

    if base not in target.parents and target != base:
        raise RuntimeError(f"Unsafe archive path: {filename}")

    return target


# ============================================================
# CRC CHECK FOR EXISTING FILES
# ============================================================

def crc32_file(path):
    crc = 0

    with open(path, "rb") as f:
        while True:
            chunk = f.read(1024 * 1024)

            if not chunk:
                break

            crc = zlib.crc32(chunk, crc)

    return crc & 0xFFFFFFFF


# ============================================================
# CHECK ZIP
# ============================================================

if not ZIP_FILE.exists():
    raise FileNotFoundError(f"Missing ZIP:\n{ZIP_FILE}")

if not zipfile.is_zipfile(ZIP_FILE):
    raise RuntimeError("Downloaded ImageNet file is not a valid ZIP.")

print("ImageNet ZIP found and readable.")


# ============================================================
# SAFE + RESUMABLE EXTRACTION
# ============================================================

with zipfile.ZipFile(ZIP_FILE, "r") as z:

    # Only keep ImageNet train, val, and required metadata
    members = []

    for member in z.infolist():

        if member.is_dir():
            continue

        name = member.filename.replace("\\", "/")

        if (
            name.startswith(KEEP_PREFIXES)
            or name in KEEP_FILES
        ):
            members.append(member)

    total_bytes = sum(
        member.file_size for member in members
    )

    print(f"Files to process: {len(members):,}")

    extracted = 0
    skipped = 0
    repaired = 0

    with tqdm(
        total=total_bytes,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc="Extracting ImageNet"
    ) as pbar:

        for member in members:

            target = safe_target(
                IMAGENET_DIR,
                member.filename
            )

            target.parent.mkdir(
                parents=True,
                exist_ok=True
            )

            # =================================================
            # CHECK EXISTING FILE
            # =================================================

            if target.exists():

                # Wrong size = incomplete/corrupt
                if target.stat().st_size != member.file_size:
                    repaired += 1

                else:
                    # Correct size: also verify CRC
                    if crc32_file(target) == member.CRC:
                        skipped += 1
                        pbar.update(member.file_size)
                        continue

                    # Same size but bad contents
                    repaired += 1


            # =================================================
            # EXTRACT TO TEMP FILE FIRST
            # =================================================

            temp_target = target.with_suffix(
                target.suffix + ".part"
            )

            if temp_target.exists():
                temp_target.unlink()

            try:

                with z.open(member) as src, open(
                    temp_target, "wb"
                ) as dst:

                    while True:
                        chunk = src.read(1024 * 1024)

                        if not chunk:
                            break

                        dst.write(chunk)
                        pbar.update(len(chunk))

                # Only rename after successful extraction
                temp_target.replace(target)

                extracted += 1

            except Exception:

                # Remove incomplete temporary file
                if temp_target.exists():
                    temp_target.unlink()

                raise


print("\nExtraction finished.")
print(f"New files:      {extracted:,}")
print(f"Already valid:  {skipped:,}")
print(f"Repaired files: {repaired:,}")

ImageNet ZIP found and readable.
Files to process: 1,331,169


Extracting ImageNet: 100%|██████████| 143G/143G [1:58:23<00:00, 21.6MB/s]    


Extraction finished.
New files:      0
Already valid:  1,331,169
Repaired files: 0


integrity check imgnet 1k

In [25]:
ROOT = Path("datasets/imagenet")
TRAIN = ROOT / "ILSVRC/Data/CLS-LOC/train"
VAL = ROOT / "ILSVRC/Data/CLS-LOC/val"

random.seed(42)

EXPECTED_TRAIN = 1_281_167
EXPECTED_VAL = 50_000
EXPECTED_CLASSES = 1_000

# 1. Check folders
assert TRAIN.is_dir(), f"Missing: {TRAIN}"
assert VAL.is_dir(), f"Missing: {VAL}"

class_dirs = sorted(
    p for p in TRAIN.iterdir() if p.is_dir()
)

# 2. Count training images and collect samples
train_count = 0
samples = []
empty_classes = []

for folder in tqdm(class_dirs, desc="Checking train classes"):
    images = [
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() == ".jpeg"
    ]

    train_count += len(images)

    if images:
        samples.append(random.choice(images))
    else:
        empty_classes.append(folder.name)

# 3. Check validation images
val_images = [
    p for p in VAL.iterdir()
    if p.is_file() and p.suffix.lower() == ".jpeg"
]

val_count = len(val_images)

samples += random.sample(
    val_images, min(100, val_count)
)

# 4. Check whether sampled images are readable
corrupted = []

for path in tqdm(samples, desc="Testing images"):
    try:
        with Image.open(path) as img:
            img.load()
    except Exception as e:
        corrupted.append((str(path), str(e)))

# 5. Results
print("\nIMAGE COUNTS")
print(f"Train:   {train_count:,} / {EXPECTED_TRAIN:,}")
print(f"Val:     {val_count:,} / {EXPECTED_VAL:,}")
print(f"Classes: {len(class_dirs):,} / {EXPECTED_CLASSES:,}")

print("\nIMAGE INTEGRITY")
print(f"Sampled:       {len(samples):,}")
print(f"Unreadable:    {len(corrupted)}")
print(f"Empty classes: {len(empty_classes)}")

for path, error in corrupted[:10]:
    print("CORRUPTED:", path, error)

complete = (
    train_count == EXPECTED_TRAIN
    and val_count == EXPECTED_VAL
    and len(class_dirs) == EXPECTED_CLASSES
    and not empty_classes
    and not corrupted
)

print("\nQuick check passed:", complete)

Testing images: 100%|██████████| 1100/1100 [00:05<00:00, 193.54it/s]


IMAGE COUNTS
Train:   1,281,167 / 1,281,167
Val:     50,000 / 50,000
Classes: 1,000 / 1,000

IMAGE INTEGRITY
Sampled:       1,100
Unreadable:    0
Empty classes: 0

Quick check passed: True


In [26]:
# DEEPER CHECK CHECK LATER 
ZIP_FILE = ROOT / "imagenet-object-localization-challenge.zip"

def crc32_file(path):
    crc = 0

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            crc = zlib.crc32(chunk, crc)

    return crc & 0xFFFFFFFF

missing = []
wrong_size = []
bad_crc = []
checked = 0

with zipfile.ZipFile(ZIP_FILE) as z:

    members = [
        m for m in z.infolist()
        if not m.is_dir()
        and (
            m.filename.startswith(
                "ILSVRC/Data/CLS-LOC/train/"
            )
            or m.filename.startswith(
                "ILSVRC/Data/CLS-LOC/val/"
            )
        )
    ]

    if not members:
        raise RuntimeError("No ImageNet images found in ZIP.")

    for member in tqdm(
        members,
        desc="Verifying ImageNet",
        unit="file"
    ):
        target = (ROOT / member.filename).resolve()

        if not target.is_relative_to(ROOT.resolve()):
            raise RuntimeError("Unsafe archive path.")

        if not target.is_file():
            missing.append(str(target))
            continue

        if target.stat().st_size != member.file_size:
            wrong_size.append(str(target))
            continue

        if crc32_file(target) != member.CRC:
            bad_crc.append(str(target))
            continue

        checked += 1

print("\nFULL INTEGRITY REPORT")
print("Valid files:", checked)
print("Missing:", len(missing))
print("Wrong size:", len(wrong_size))
print("Bad CRC:", len(bad_crc))

for category in (missing, wrong_size, bad_crc):
    for path in category[:5]:
        print(path)

print(
    "\nAll archive images verified:",
    checked == len(members)
)

Verifying ImageNet: 100%|██████████| 1331167/1331167 [1:37:08<00:00, 228.40file/s]


FULL INTEGRITY REPORT
Valid files: 1331167
Missing: 0
Wrong size: 0
Bad CRC: 0

All archive images verified: True


# download ade20k

In [27]:
# %pip install -U huggingface_hub hf_xet

In [28]:
ADE_DIR = Path("datasets/ade20k")
ADE_DIR.mkdir(parents=True, exist_ok=True)

ZIP_FILE = ADE_DIR / "ADEChallengeData2016.zip"

EXPECTED_SHA256 = (
    "7ff1be44964418441f542a7cc1e1a650"
    "e7dc0fc275f5d23252bc9bbdbc977b29"
)

def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# Skip an already verified download
if ZIP_FILE.exists() and sha256_file(ZIP_FILE) == EXPECTED_SHA256:
    print("Existing ZIP verified. Skipping download.")

else:
    print("Downloading ADE20K from Hugging Face...")

    downloaded = hf_hub_download(
        repo_id="zbwxp/ade",
        filename="ADEChallengeData2016.zip",
        repo_type="dataset",
        local_dir=str(ADE_DIR),
    )

    ZIP_FILE = Path(downloaded)

# Verify the downloaded file
assert sha256_file(ZIP_FILE) == EXPECTED_SHA256, (
    "SHA-256 mismatch. The archive is not the expected mirror file."
)

with zipfile.ZipFile(ZIP_FILE) as z:
    bad_file = z.testzip()

assert bad_file is None, f"Corrupted ZIP member: {bad_file}"

print("Download complete. ZIP integrity verified.")

Existing ZIP verified. Skipping download.
Download complete. ZIP integrity verified.


In [29]:
ROOT = ADE_DIR.resolve()

def file_crc(path):
    crc = 0
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            crc = zlib.crc32(chunk, crc)
    return crc & 0xFFFFFFFF

extracted = skipped = repaired = 0

with zipfile.ZipFile(ZIP_FILE) as z:
    members = [m for m in z.infolist() if not m.is_dir()]

    if len({m.filename for m in members}) != len(members):
        raise RuntimeError("Duplicate archive paths detected.")

    total = sum(m.file_size for m in members)

    with tqdm(
        total=total,
        unit="B",
        unit_scale=True,
        unit_divisor=1024,
        desc="Extracting ADE20K"
    ) as bar:

        for member in members:
            target = (ROOT / member.filename).resolve()

            if not target.is_relative_to(ROOT):
                raise RuntimeError(f"Unsafe path: {member.filename}")

            target.parent.mkdir(parents=True, exist_ok=True)

            if target.exists():
                if (
                    target.stat().st_size == member.file_size
                    and file_crc(target) == member.CRC
                ):
                    skipped += 1
                    bar.update(member.file_size)
                    continue

                repaired += 1

            temp = target.with_name(target.name + ".part")

            try:
                with z.open(member) as src, open(temp, "wb") as dst:
                    for chunk in iter(lambda: src.read(1024 * 1024), b""):
                        dst.write(chunk)
                        bar.update(len(chunk))

                if temp.stat().st_size != member.file_size:
                    raise RuntimeError(f"Incomplete file: {member.filename}")

                temp.replace(target)
                extracted += 1

            except Exception:
                temp.unlink(missing_ok=True)
                raise

print(f"Extracted: {extracted:,}")
print(f"Already valid: {skipped:,}")
print(f"Repaired: {repaired:,}")

Extracting ADE20K: 100%|██████████| 923M/923M [03:03<00:00, 5.29MB/s] 

Extracted: 0
Already valid: 44,422
Repaired: 0


In [30]:
ROOT = ADE_DIR / "ADEChallengeData2016"

folders = {
    "Train images": ("images/training", "*.jpg", 20210),
    "Validation images": ("images/validation", "*.jpg", 2000),
    "Train masks": ("annotations/training", "*.png", 20210),
    "Validation masks": ("annotations/validation", "*.png", 2000),
}

all_complete = True

for label, (folder, pattern, expected) in folders.items():
    count = sum(1 for _ in (ROOT / folder).glob(pattern))
    print(f"{label}: {count:,} / {expected:,}")
    all_complete &= count == expected

print("Dataset complete:", all_complete)

Train images: 20,210 / 20,210
Validation images: 2,000 / 2,000
Train masks: 20,210 / 20,210
Validation masks: 2,000 / 2,000
Dataset complete: True


integrity checks 

In [2]:

ROOT = Path("datasets/ade20k")
DATA = ROOT / "ADEChallengeData2016"

EXPECTED_TRAIN = 20_210
EXPECTED_VAL = 2_000

random.seed(42)

def check_split(split, expected):
    image_dir = DATA / "images" / split
    mask_dir = DATA / "annotations" / split

    assert image_dir.is_dir(), f"Missing: {image_dir}"
    assert mask_dir.is_dir(), f"Missing: {mask_dir}"

    images = {
        p.stem: p for p in image_dir.glob("*.jpg")
    }
    masks = {
        p.stem: p for p in mask_dir.glob("*.png")
    }

    print(f"\n{split.upper()}")
    print(f"Images: {len(images):,} / {expected:,}")
    print(f"Masks:  {len(masks):,} / {expected:,}")

    # Ensure every image has a corresponding mask
    missing_masks = images.keys() - masks.keys()
    missing_images = masks.keys() - images.keys()

    print("Images without masks:", len(missing_masks))
    print("Masks without images:", len(missing_images))

    # Test up to 100 image-mask pairs
    pairs = sorted(images.keys() & masks.keys())
    samples = random.sample(pairs, min(100, len(pairs)))

    corrupted = []

    for name in tqdm(samples, desc=f"Testing {split}"):
        try:
            with Image.open(images[name]) as img:
                img.load()
                image_size = img.size

            with Image.open(masks[name]) as mask:
                mask.load()
                mask_size = mask.size

            if image_size != mask_size:
                corrupted.append((name, "Image/mask size mismatch"))

        except Exception as e:
            corrupted.append((name, str(e)))

    print("Sampled pairs:", len(samples))
    print("Unreadable/mismatched pairs:", len(corrupted))

    for name, error in corrupted[:10]:
        print(name, error)

    return (
        len(images) == expected
        and len(masks) == expected
        and not missing_masks
        and not missing_images
        and not corrupted
    )


train_ok = check_split("training", EXPECTED_TRAIN)
val_ok = check_split("validation", EXPECTED_VAL)

print("\nADE20K quick check passed:", train_ok and val_ok)

NameError: name 'Path' is not defined

In [4]:

# ============================================================
# CONFIGURATION
# ============================================================

ADE20K_ROOT = Path("datasets/ade20k").resolve()

DATA = ADE20K_ROOT / "ADEChallengeData2016"
ZIP_FILE = ADE20K_ROOT / "ADEChallengeData2016.zip"


# ============================================================
# CRC CHECK
# ============================================================

def crc32_file(path):
    crc = 0

    with open(path, "rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            crc = zlib.crc32(chunk, crc)

    return crc & 0xFFFFFFFF


# ============================================================
# CHECK DATASET PATHS
# ============================================================

if not DATA.is_dir():
    raise FileNotFoundError(
        f"ADE20K dataset folder not found: {DATA}"
    )

if not ZIP_FILE.is_file():
    raise FileNotFoundError(
        f"ADE20K ZIP not found: {ZIP_FILE}"
    )

if not zipfile.is_zipfile(ZIP_FILE):
    raise RuntimeError("ADE20K ZIP is invalid.")


# ============================================================
# FULL INTEGRITY VERIFICATION
# ============================================================

missing = []
wrong_size = []
bad_crc = []
checked = 0

with zipfile.ZipFile(ZIP_FILE) as z:

    members = []

    for member in z.infolist():

        if member.is_dir():
            continue

        name = member.filename.replace("\\", "/")

        # Handle ZIPs with or without the top-level folder
        if name.startswith("ADEChallengeData2016/"):
            relative = name.removeprefix(
                "ADEChallengeData2016/"
            )
        else:
            relative = name

        if relative.startswith(("images/", "annotations/")):
            members.append((member, relative))

    if not members:
        raise RuntimeError(
            "No ADE20K images or masks found in ZIP."
        )

    print(f"Files to verify: {len(members):,}")

    for member, relative in tqdm(
        members,
        desc="Verifying ADE20K",
        unit="file"
    ):

        target = (DATA / relative).resolve()

        # Prevent unsafe paths
        if not target.is_relative_to(DATA.resolve()):
            raise RuntimeError(
                f"Unsafe archive path: {relative}"
            )

        # Check file existence
        if not target.is_file():
            missing.append(relative)
            continue

        # Check file size
        if target.stat().st_size != member.file_size:
            wrong_size.append(relative)
            continue

        # Check file CRC
        if crc32_file(target) != member.CRC:
            bad_crc.append(relative)
            continue

        checked += 1


# ============================================================
# FINAL REPORT
# ============================================================

print("\nFULL ADE20K INTEGRITY REPORT")

print(f"Verified files: {checked:,} / {len(members):,}")
print(f"Missing:        {len(missing):,}")
print(f"Wrong size:     {len(wrong_size):,}")
print(f"Bad CRC:        {len(bad_crc):,}")

for category in (missing, wrong_size, bad_crc):
    for path in category[:5]:
        print(path)

Files to verify: 44,420


Verifying ADE20K: 100%|██████████| 44420/44420 [04:19<00:00, 170.92file/s]


FULL ADE20K INTEGRITY REPORT
Verified files: 44,420 / 44,420
Missing:        0
Wrong size:     0
Bad CRC:        0
